# GenAI

## Contagem de Tokens

Acesse o link [Tokenizer](https://platform.openai.com/tokenizer) da OpenAI para ver como cada texto é convertido em Tokens. É possível definir qual modelo quer usar.

## Diferença de Alucinação e Cut-off

Ex. 1) Uso do multimodal no [Google AI Studio](https://aistudio.google.com/) para interagir com Webcam e discutir sobre os livros.

Ex. 2) Quem é o Campeão Paulista de Futebol Masculino da Séria A em 2025?

## Vetores e Vector Database

Os detalhes do modelo [Gemma3](https://huggingface.co/google/gemma-3-4b-it), assim como do [Stable Diffusion](https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers) podem ser acessados diretamente no site da Hugging Face.

Este é um token para o exemplo e será revogado logo após a demonstração, cada um deve ter o seu:

```
SEU TOKEN
```

Para [criar o seu token](https://huggingface.co/settings/tokens), acesse o site da Hugging Face e se cadastre gratuitamente.


In [ ]:
!pip install -U sentence-transformers git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from huggingface_hub import login

In [ ]:
login()

O modelo de ***Embedding*** utilizado nesta demonstração é o [Gemma3Embedding](https://ai.google.dev/gemma/docs/embeddinggemma?hl=pt-br), que é da Google e permite um uso leve por exigir apenas 300MB de RAM

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/embeddinggemma-300M"
model = SentenceTransformer(model_id).to(device=device)

print(f"Dispositivo (motor de processamento): {model.device}")
print(model)
print("Número total de parâmetros do modelo:", sum([p.numel() for _, p in model.named_parameters()]))

In [ ]:
palavras = ["rei", "rainha", "homem", "mulher", "computador", "celular", "maçã", "banana"]

embeddings = model.encode(palavras)

print(embeddings)
for idx, embedding in enumerate(embeddings):
  print(f"Embedding {idx+1} (shape): {embedding.shape}")

In [ ]:
print("Tarefas disponíveis no modelo:")
for name, prefix in model.prompts.items():
  print(f" {name}: \"{prefix}\"")
print("-"*80)

In [ ]:
def encontrar_similaridade():
  print("Função de Similaridade: ", model.similarity_fn_name)
  similarities = model.similarity(embeddings[0:], embeddings[0:])
  print(similarities)

  print("----------")
  for i, palavra1 in enumerate(palavras[0:]):
    print(f"*{palavra1}*")
    for j, palavra2 in enumerate(palavras[0:]):
      #print(f"*{palavra1}* com *{palavra2}* tem similaridade de {similarities.numpy()[i][j]}" )
      print(f"com *{palavra2}* tem similaridade de {similarities.numpy()[i][j]:.2f}" )
    print("----------")
embeddings = model.encode(palavras, prompt_name="STS")

encontrar_similaridade()

## Exploração de Modelos pré-treinados

Todos os [detalhes de modelos](https://huggingface.co/models) que existem na Hugging Face podem ser acessados pelo portal, ou por linha de comando.

In [ ]:
from huggingface_hub import list_models

In [ ]:
def listar_modelos_texto(limite=50):
    print(f"\nListando os {limite} modelos de 'text-generation' mais baixados")
    modelos = list(list_models(filter="text-generation", sort="downloads", limit=limite))
    print("Nome do Modelo | Quantidade de downloads")
    print("-------------------")

    for modelo in modelos:
        print(f"{modelo.modelId} | {modelo.downloads}")

listar_modelos_texto()

## Teachable Machine

O [Techable Machine](https://teachablemachine.withgoogle.com/) pode ser acessado pelo portal, ou via código.

### Baixar as imagens para treinar o modelo

Para baixar as imagens, vamos usar uma Biblioteca para download do Google Image, e depois, usaremos essas imagens para treinar o modelo.

In [ ]:
print("Instalando a biblioteca simple_image_download...")
!pip install -q simple_image_download
print("Biblioteca simple_image_download instalada com sucesso!")

In [ ]:
import os
import zipfile

from google.colab import files
from simple_image_download import simple_image_download


print("Bibliotecas carregadas com sucesso!")

In [ ]:
def fazer_busca(num_imagens, termo_busca):
  print(f"\nIniciando o download de {num_imagens} imagens de '{termo_busca}'...")

  try:
      downloader = simple_image_download.simple_image_download()
      downloader.download(termo_busca, num_imagens)
      nome_pasta = termo_busca

      print("Download concluído com sucesso!")
      print(f"As suas imagens estão salvas na pasta:")
      print(f"-> simple_images/{nome_pasta}")

      return f"/content/simple_images/{nome_pasta}"

  except Exception as e:
      print(f"\n Ocorreu um erro durante o download.")
      print(f"Detalhe do erro: {e}")

In [ ]:
def renomear_pasta_para_underscore(caminho_da_pasta):
    print(f"Tentando renomear: '{caminho_da_pasta}'")

    if not os.path.exists(caminho_da_pasta):
        print(f"Erro: A pasta '{caminho_da_pasta}' não foi encontrada.")
        return

    if not os.path.isdir(caminho_da_pasta):
        print(f"Erro: O caminho '{caminho_da_pasta}' é um arquivo, não uma pasta.")
        return

    diretorio_pai, nome_antigo = os.path.split(caminho_da_pasta)
    nome_novo = nome_antigo.replace(' ', '_')
    novo_caminho_da_pasta = os.path.join(diretorio_pai, nome_novo)

    if os.path.exists(novo_caminho_da_pasta):
        print(f"Erro: Já existe um arquivo ou pasta com o nome '{novo_caminho_da_pasta}'.")
        print("A renomeação foi cancelada para evitar conflitos.")
        return

    try:
        os.rename(caminho_da_pasta, novo_caminho_da_pasta)

        print("Pasta renomeada com sucesso!")
        print(f"De: {caminho_da_pasta}")
        print(f"Para: {novo_caminho_da_pasta}")
        return novo_caminho_da_pasta

    except OSError as e:
        print(f"Erro inesperado ao tentar renomear a pasta.")
        print(f"Detalhe do erro: {e}")
        return

In [ ]:
def compactar_e_baixar_imagens_da_pasta(pasta_entrada):
    print(f"Analisando a pasta: '{pasta_entrada}'")

    if not os.path.isdir(pasta_entrada):
        print(f"Erro: A pasta '{pasta_entrada}' não foi encontrada.")
        print("Verifique se o nome está correto e se o script anterior de download rodou.")
        return

    tipos_de_imagem = ('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp')

    nome_base_pasta = os.path.basename(pasta_entrada)
    nome_arquivo_zip = f"{nome_base_pasta}_imagens.zip"

    imagens_para_compactar = []

    print("Procurando por arquivos de imagem...")
    for arquivo in os.listdir(pasta_entrada):
        if arquivo.lower().endswith(tipos_de_imagem):
            caminho_completo = os.path.join(pasta_entrada, arquivo)
            imagens_para_compactar.append((caminho_completo, arquivo))

    if not imagens_para_compactar:
        print(f"Aviso: Nenhuma imagem encontrada na pasta '{pasta_entrada}'.")
        return

    print(f"Encontradas {len(imagens_para_compactar)} imagens. Iniciando compactação...")

    try:
        with zipfile.ZipFile(nome_arquivo_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
            for caminho_completo, nome_do_arquivo in imagens_para_compactar:
                zf.write(caminho_completo, arcname=nome_do_arquivo)

        print(f"Arquivo '{nome_arquivo_zip}' criado com sucesso.")

        print(f"Iniciando o download de '{nome_arquivo_zip}'.")
        print("Aguarde a janela de download do seu navegador...")
        files.download(nome_arquivo_zip)

    except FileNotFoundError:
        print(f"Erro: O arquivo não foi encontrado durante a compactação.")
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")

In [ ]:
def comecar():
  termo_busca = input("Qual imagem você quer baixar? (Ex: 'carros esportivos'): ")
  num_imagens = int(input("Quantas imagens você quer baixar? (Ex: 5): "))

  pasta_original = fazer_busca(num_imagens, termo_busca)
  pasta_renomeada = renomear_pasta_para_underscore(pasta_original)

  compactar_e_baixar_imagens_da_pasta(pasta_renomeada)

In [ ]:
# 30 imagens
# corte de cabelo masculino
# homem com boné

comecar()

### Rodar no Google Colab

Agora que já escolhemos os termos, baixamo as imagens, treinamos e testamos o modelo no portal, é hora de testar via código.

In [ ]:
import tensorflow as tf
from keras.layers import TFSMLayer
from PIL import Image, ImageOps
import numpy as np

In [ ]:
model = TFSMLayer(
    "/content/saved_model",
    call_endpoint="serving_default"
)

class_names = open("/content/labels.txt", "r").readlines()

In [ ]:
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

# imagemOriginal = Image.open("/content/sem-bone.jpeg").convert("RGB")
imagemOriginal = Image.open("/content/com-bone.jpeg").convert("RGB")
image = imagemOriginal

size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)

In [ ]:
image_array = np.asarray(image)

normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1

data[0] = normalized_image_array

prediction = model.predict(data)
index = np.argmax(prediction)
class_name = class_names[index]
confidence_score = prediction[0][index]

In [ ]:
largura_original, altura_original = imagemOriginal.size
nova_largura = 200
nova_altura = int((altura_original / largura_original) * nova_largura)

imagemOriginal_redimensionada = imagemOriginal.resize((nova_largura, nova_altura))

display(imagemOriginal_redimensionada)

print("Classe:", class_name[2:], end="")
print("Confiança:", confidence_score)

## Implementação com TensorFlow


Este [código foi criado inicialmente](https://github.com/diegonogare/MachineLearning/blob/main/5.5-MNIST_GAN.ipynb) durante uma disciplina do meu doutorado. Você pode acompanhar toda o código gerado na disciplina no meu Github.

## Experimentação de diferentes técnicas de prompts

O objetivo desta demonstração é tratar a IA como um conjunto de ferramentas de precisão.

Usaremos um material que conheço muito bem: alguns dos meus artigos científicos da época do doutorado. Este material (como qualquer outro texto científico) é, por natureza, denso, complexo e escrito para um público muito específico. Vamos fazer isso ficar acessível!

O que faremos:

*   Aplicar tarefas fundamentais da IA Generativa (Resumo, Tradução, Classificação, Extração, Análise de Sentimento, P&R);

*   Aumentar a complexidade de suas instruções, passando de prompts Zero-Shot para prompts de Instrução Detalhada e Persona;


### Tradução

Traduza o ***abstract*** (resumo) deste artigo para o português;


```
### **Persona**
Atue como um **Tradutor Técnico Especializado** em Inteligência Artificial e publicações acadêmicas.
Sua função é realizar uma tradução precisa, mantendo a fidelidade ao conteúdo científico e utilizando a terminologia técnica em português de forma adequada e coerente com o campo da Inteligência Artificial.

### **Estilo**
Formal, conciso e com alta precisão terminológica.

### **Tom**
Objetivo e profissional.

### **Atividade**
Localize o **Abstract** (Resumo) do artigo fornecido e o traduza integralmente para o português.
Garanta que a tradução reflita o significado original com clareza e aderência à linguagem científica.
O Abstract está localizado no início do artigo..

### **Formato de Saída**
A saída deve conter apenas o Abstract traduzido, formatado como um único parágrafo.
```

### Simplificação

Agora, pegue esta tradução e peça para simplificar. Pode direcionar o LLM para que tenha no máximo 100 palavras e evite jargões técnicos



```
### Persona
Atue como um **Mentor de Carreira em IA/ML** (Inteligência Artificial e Machine Learning).
Sua tarefa é simplificar conceitos complexos, tornando-os acessíveis e relevantes para um profissional que está fazendo uma transição de carreira para a área de IA.

### Estilo
Didático, claro e altamente conciso.

### Tom
Encorajador e informativo.

### Atividade
Analise o artigo fornecido e crie um resumo de, no **máximo, 100 palavras**. O objetivo é explicar o que está no resumo, por que ele é importante (o problema que resolve), e qual é a principal contribuição do artigo.
Evite jargões técnicos complexos ou, se necessário, explique-os de forma muito simples.

### Formato de Saída
Um único parágrafo conciso, sem bullet points, que não ultrapasse 100 palavras.
O resumo deve ser fácil de entender para um iniciante em IA.

```

### Resumo

Use a seção **Introdução** ou **Conclusão** do artigo

NÃO peça apenas para resumir, você deve projetar o resumo.
Declare de forma explícita que quer saber qual problema o artigo resolve,  qual principal contribuição, qual resultado obtido...



```
### Persona
Atue como um **Analista de Conteúdo Acadêmico** rigoroso e detalhista.
Sua função é extrair os elementos centrais do artigo para construir um resumo estruturado, focado em impacto e contribuição científica e prática.

### Estilo
Sintético, estruturado e altamente informativo.

### Tom
Analítico e objetivo.

### Atividade
Utilize **exclusivamente** o conteúdo das seções Introdução e Conclusão do artigo fornecido para construir um resumo projetado, que deve abordar os seguintes pontos de forma clara e sequencial:

1.  **Problema Resolvido:** Qual é o desafio ou a lacuna (gap) principal que o artigo busca solucionar?
2.  **Principal Contribuição:** Qual é a proposta central do artigo (o novo modelo, arquitetura ou abordagem)?
3.  **Resultados Obtidos (Impacto Prático):** Cite um resultado quantificável ou um impacto prático significativo obtido com a aplicação do modelo (por exemplo, redução de tempo, aumento de escala).

Certifique-se de que cada ponto do resumo projetado seja suportado por evidências encontradas nas seções especificadas.

### Formato de Saída
A saída deve ser uma lista com bullet points, onde cada ponto da lista corresponde a um dos tópicos solicitados (Problema, Contribuição, Resultados), garantindo que os dados extraídos sejam concisos.

**Exemplo de Formato:**
* **Problema Resolvido:** [Fato extraído do texto]
* **Principal Contribuição:** [Fato extraído do texto]
* **Resultados Obtidos:** [Fato extraído do texto]

```


### Análise de Sentimentos

Mande o LLM procurar as citações. Classifique o sentimento dessa citação como: [Crítico], [Neutro/Descritivo] ou [Complementar/Baseado em] e justifique sua resposta.


```

### Persona
Atue como um **Analista Crítico de Literatura Científica**.
Sua função é examinar o uso de referências bibliográficas (citações) dentro do texto e classificá-las de acordo com o papel que desempenham no argumento do artigo.

### Estilo
Analítico, detalhado e estruturado.

### Tom
Objetivo e acadêmico.

### Atividade
Selecione uma seção do artigo (por exemplo, a Seção **RELATED WORKS** ou a Seção **EVALUATION**) que contenha múltiplas citações. Para cada citação (ou grupo de citações) em uma frase, classifique o **sentimento** da citação em relação ao texto do artigo, usando estritamente uma das seguintes categorias:

1.  **[Crítico]:** A citação é usada para apontar um problema, uma limitação, uma falha ou uma lacuna no estado da arte que o artigo proposto busca resolver;
2.  **[Neutro/Descritivo]:** A citação é usada para definir um conceito, descrever um fato estabelecido, listar tecnologias, ou apresentar um contexto factual sem um juízo de valor explícito;
3.  **[Complementar/Baseado em]:** A citação é usada para fornecer a base teórica, o suporte, a evidência ou a fonte de uma afirmação, metodologia ou resultado que o artigo está utilizando ou construindo.

Para cada classificação, forneça uma **justificativa concisa** baseada no contexto da frase onde a citação aparece.

### Formato de Saída
A saída deve ser uma tabela Markdown com três colunas:

| Frase e Citação | Classificação | Justificativa |
| :--- | :--- | :--- |
| *[Frase completa contendo a citação]* | *[Categoria]* | *[Breve explicação do porquê]* |
| ... | ... | ... |
| ... | ... | ... |
```



### Classificação de Assuntos


Peça para o LLM analisar o ***abstract*** e dizer para qual destas áreas de atuação o assunto é mais relevante: *MLOps*, *Engenharia de Dados*, *Desenvolvimento Web* ou *Desenvolvimento Mobile*? Justifique em um parágrafo.


```
### Persona
Atue como um **Especialista em Classificação de Domínios de TI**, focado em avaliar a relevância de artigos científicos para áreas específicas da tecnologia.

### Estilo
Analítico, direto e bem fundamentado.

### Tom
Objetivo e autoritário.

### Atividade
Analise o **Abstract** do artigo fornecido. Com base nesse texto, determine para qual das seguintes áreas de atuação o conteúdo do artigo é **mais relevante**:

1.  **MLOps**;
2.  **Engenharia de Dados**;
3.  **Desenvolvimento Web**;
4.  **Desenvolvimento Mobile**.

Após a escolha, redija um parágrafo único de justificativa, mencionando os termos-chave do Abstract que fundamentam sua decisão. A justificativa deve explicar por que a área escolhida é a mais relevante e por que as outras são secundárias ou não são o foco principal.

### Formato de Saída
A saída deve conter o nome da área mais relevante em negrito, seguido por um traço, e então o parágrafo de justificativa.

**Exemplo de Formato:**
**Área relevante** - Justificativa...
```



### Extração de entidades simples

Peça para o LLM listar todas as ferramentas/software, linguagens de programação ou provedores de computação em nuvem mencionados no texto...

Use a seção **Metodologia** ou *Experimentos* para extrair esta lista de artefatos.




```
### Persona
Atue como um **Bibliotecário de Tecnologia** especializado em ecossistemas de IA.
Sua função é realizar uma varredura exaustiva no artigo para catalogar todas as menções a ferramentas, plataformas, linguagens e provedores de infraestrutura.

### Estilo
Sistemático, exaustivo e organizado.

### Tom
Factual e informativo.

### Atividade
Percorra o artigo e extraia todas as menções a:
1.  **Ferramentas/Software & Frameworks de MLOps/ML/Data Science** (ex: MLFlow, Kubeflow, Pandas, Scikit-learn);
2.  **Linguagens de Programação** (ex: Python, R, Java);
3.  **Provedores de Computação em Nuvem e Plataformas Específicas** (ex: AWS, Azure ML, GCP).

Após a coleta, organize cada item em uma tabela, classificando-o em uma das categorias acima e indicando a seção do artigo onde foi mencionado para referência rápida.

### Formato de Saída
A saída deve ser uma tabela Markdown com três colunas: **Tecnologia/Nome**, **Categoria** e **Seção de Menção**.

| Tecnologia/Nome | Categoria | Seção de Menção |
| :--- | :--- | :--- |
| *[Nome da Tecnologia]* | *[Ferramentas/Software, Linguagens ou Provedores de Nuvem]* | *[Ex: I. INTRODUCTION, III. METHODOLOGY]* |
| ... | ... | ... |
| ... | ... | ... |
```



### Extração de entidades estruturadas


Peça para que o LLM responda APENAS com um objeto em tabela. Se uma informação não for encontrada, retorne “NÃO ENCONTREI”....

Use a mesma premissa do exercício anterior para extrair a lista de artefatos.


```
### Persona
Atue como um **Bibliotecário de Tecnologia** especializado em ecossistemas de IA.
Sua função é realizar uma varredura exaustiva no artigo para catalogar todas as menções a ferramentas, plataformas, linguagens e provedores de infraestrutura.

### Estilo
Sistemático, exaustivo e organizado.

### Tom
Factual e informativo.

### Atividade
Percorra o artigo e extraia todas as menções a:
1.  **Ferramentas/Software & Frameworks de MLOps/ML/Data Science** (ex: MLFlow, Kubeflow, Pandas, Scikit-learn).
2.  **Linguagens de Programação** (ex: Python, R, Java).
3.  **Provedores de Computação em Nuvem e Plataformas Específicas** (ex: AWS, Azure ML, GCP).

Após a coleta, organize cada item em uma tabela, classificando-o em uma das categorias acima e indicando a seção do artigo onde foi mencionado para referência rápida. Se uma informação não for encontrada no artigo para um campo, retorne **“NÃO ENCONTREI”**.

### Formato de Saída
A saída deve ser **APENAS** uma tabela Markdown com três colunas: **Tecnologia/Nome**, **Categoria** e **Seção de Menção**.

| Tecnologia/Nome | Categoria | Seção de Menção |
| :--- | :--- | :--- |
| *[Nome da Tecnologia]* | *[Ferramentas/Software, Linguagens ou Provedores de Nuvem]* | *[Ex: I. INTRODUCTION, III. METHODOLOGY]* |
| ... | ... | ... |
| ... | ... | ... |

```



### Ancoragem em Contexto - Pergunta existente


Peça para que o LLM use somente o documento fornecido para responder à pergunta:

`Faça uma pergunta MUITO específica sobre a metodologia ou resultados do artigo`



```
### Persona
Atue como um **professor** da área de IA.
Sua função é ensinar de forma didática e clara conceitos específicos.

### Estilo
Organizado.

### Tom
Informativo.

### Atividade
Use exclusivamente o documento fornecido para responder à pergunta [Quais são os três pilares fundamentais do MLOps?]

```



### Ancoragem em Contexto - Teste de Estress


Seguindo a mesma premissa do contexto anterior

`Faça uma pergunta sobre algo que NÃO ESTÁ no artigo `

Coloque algo completamente apartado do assunto do artigo, como futebol ou alimentação




```

### Persona
Atue como um **professor** da área de IA.
Sua função é ensinar de forma didática e clara conceitos específicos.

### Estilo
Organizado.

### Tom
Informativo.

### Atividade
Use exclusivamente o documento fornecido para responder à pergunta [Quando foi a última vez que a seleção de futebol masculino do Brasil ganhou uma Copa do Mundo?]

```

## Geração de Texto e de Imagem com Python

### Importante

Esta parte deve ser executada independente de ser modelo de linguagem ou modelo de imagem.

Como o ambiente do Colab é gratuito e limitado, é recomendado que faça um dos exemplos com GPU e depois reinicie o ambiente para liberar memória.

In [ ]:
# Instalação das bibliotecas necessárias da Hugging Face
#!pip install transformers torch -q

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, AutoProcessor, Gemma3ForConditionalGeneration
from huggingface_hub import notebook_login
#from diffusers import StableDiffusion3Pipeline
from diffusers import StableDiffusionPipeline

from IPython.display import display
from PIL import Image

import torch
import requests

Os detalhes do modelo [Gemma3](https://huggingface.co/google/gemma-3-4b-it), assim como do [Stable Diffusion](https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers) podem ser acessados diretamente no site da Hugging Face.

Este é um token para o exemplo e será revogado logo após a demonstração, cada um deve ter o seu:

```
SEU TOKEN
```

Para [criar o seu token](https://huggingface.co/settings/tokens), acesse o site da Hugging Face e se cadastre gratuitamente.

In [ ]:
notebook_login()

### Geração de texto


In [ ]:
### Aproximadamente 3 minutos para carregamento do ambiente
pipe = pipeline(
    task="text-generation",
    model="google/gemma-3-4b-it",
    device="cuda",
    torch_dtype=torch.bfloat16
)

In [ ]:
prompt = """Aja como um desenvolvedor de software que está começando o na área de Inteligência Artificial.
Você quer colocar um modelo em produção, mas não quer fazer isso manualmente.
Apresente um roteiro com até 500 palavras de passos necessários para colocar um modelo de IA em produção de forma automatizada."""

In [ ]:
output = pipe(prompt, max_new_tokens=1024)
print(output[0]['generated_text'])

### Análise de imagem

In [ ]:
### Aproximadamente 3 minutos para carregamento do ambiente
pipe = pipeline(
    "image-text-to-text",
    model="google/gemma-3-4b-it",
    device="cuda",
    torch_dtype=torch.bfloat16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Device set to use cuda


In [ ]:
# https://exame.com/bussola/vibe-coding-como-programar-sem-saber-codigo-e-os-riscos-que-isso-traz-para-empresas/
# https://www.baguete.com.br/noticias/vibe-coding-o-novo-jeito-de-criar-software
# https://media.licdn.com/dms/image/v2/D4D22AQH_CKmI2UMaFQ/feedshare-shrink_800/B4DZcrYF1NGwAk-/0/1748779426028?e=2147483647&v=beta&t=p8mh2eewk3dD_1Nk7qW9KfSZl_yECkO80bzp8o0xrTA

messages = [
    {
        #Aqui está o System Prompt
        "role": "system",
        #"content": [{"type": "text", "text": "Você é um assistente de design que irá responder apenas sobre os personagens da imagem."}]
        #"content": [{"type": "text", "text": "Você é um agendador de compromissos, se atente apenas sobre datas e atividades."}]
        "content": [{"type": "text", "text": "Você é profissional senior de criação de conteúdo, faça a análise da disposição dos elementos. Formate a resposta em 50 palavras"}]
    },
    {
        #Aqui está o Prompt do usuário
        "role": "user",
        "content": [
            {"type": "image", "url": "https://media.licdn.com/dms/image/v2/D4D22AQH_CKmI2UMaFQ/feedshare-shrink_800/B4DZcrYF1NGwAk-/0/1748779426028?e=2147483647&v=beta&t=p8mh2eewk3dD_1Nk7qW9KfSZl_yECkO80bzp8o0xrTA"},
            {"type": "text", "text": "O que você vê na imagem?"}
        ]
    }
]

In [ ]:
output = pipe(text=messages, max_new_tokens=200)
print(output[0]["generated_text"][-1]["content"])

A imagem apresenta uma disposição equilibrada, com o rosto do profissional Diego Nogali ocupando destaque na parte superior. A agenda de palestras é centralizada abaixo, em formato de lista, garantindo legibilidade. Cores contrastantes e o logotipo reforçam a identidade da marca, promovendo uma comunicação clara e direta.


### Geração de imagem

In [ ]:
'''
pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    torch_dtype=torch.float16
)
pipe.to("cuda")
'''
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)
pipe.to("cuda")


In [ ]:
'''
import gc
gc.collect()

pipe.enable_model_cpu_offload()

torch.cuda.empty_cache()
'''

In [ ]:
prompt = "A car in the street with buildings"
# "A rose flower in the garden"


In [ ]:
images = pipe(
    prompt=prompt,
    num_inference_steps=28,
    guidance_scale=7.0,
    width=512,
    height=512,
).images

In [ ]:
display(images[0])